# Get preds on every hyper-param combination across folds/repeats

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
import subprocess
from src.config import BASE_PATH, SEED
from src.utils import ALPHA_LIST
from src.swarm import (
    make_perm_level,
    run_perm_swarm,
)

## Set Up

In [ ]:
SEG_TYPES = [
    "turns",
    "pt_noshort",
]
MODEL_NAMES = [
    "Llama-3.3-70B-Instruct",
]
IMPACT_TYPES = [
    "self",
    "parent",
]
PROB_AGG_TYPES = [
    "block",
    "max",
    "avg",
    "proba-limited",
    "binary",
]
DESC_TYPES = [
    "original",
    "expanded",
    "combined",
]
LINEAR_TYPES = ["ridge"]
SAVE_DIR = BASE_PATH / "perm_res" / "preds"

START_PERM = 0
END_PERM = 1000  # non-inclusive
INCLUDE_BASELINE = True

In [ ]:
param_dict = {
    "layers": [1, 2, 3],
    "agg": PROB_AGG_TYPES,
    "llama": MODEL_NAMES,
    "desc": DESC_TYPES,
    "seg": SEG_TYPES,
    "alpha": ALPHA_LIST,
}
config_dict = {
    "max_iter": 50000,
    "seed": SEED,
    "n_repeats": 10,
    "split_tol": 0.2,
    "n_splits": 3,
    "n_jobs": 1,
}

## Outputs global file for each perm (preferred)

In [ ]:
SWARM_DIR = BASE_PATH / "swarm_dir" / "preds"
cmd_dir = SWARM_DIR / "commands"

In [ ]:
total_cmd_list = []
for impact_type in IMPACT_TYPES:
    swarm_path = cmd_dir / f"{impact_type}.swarm"
    swarm_path.parent.mkdir(exist_ok=True, parents=True)
    if swarm_path.exists():
        swarm_path.unlink()
    impact_cmd_list = []
    for perm_dir in SAVE_DIR.iterdir():
        perm_name = perm_dir.name
        if perm_name != "baseline":
            # Set starting and end perm numbers to control where we are permuting
            perm_num = int(perm_name.split("_")[1].strip(""))
            if (perm_num < START_PERM) or (perm_num >= END_PERM):
                continue
            shuffle = "text"
        else:
            if not INCLUDE_BASELINE:
                continue
            shuffle = "none"
        df_dir = BASE_PATH / "data"
        scratch_dir = "/lscratch/$SLURM_JOB_ID/"
        # scratch_dir = BASE_PATH / "scratch"
        cmd = make_perm_level(
            param_dict=param_dict,
            # pass copy w/ shuffle dynamically set
            config_dict={
                **config_dict,
                "shuffle": shuffle,
            },
            df_dir=df_dir,
            scratch_dir=scratch_dir,
            perm_dir=perm_dir,
            impact_type=impact_type,
        )
        impact_cmd_list.append(cmd)
    swarm_path.write_text("\n".join(impact_cmd_list))
    total_cmd_list += impact_cmd_list
print(f"Number of commands: {len(total_cmd_list)}")

Run sequentially locally

In [ ]:
# for iteration, cmd in enumerate(total_cmd_list):
#     print(f"{iteration +1}/{len(total_cmd_list)}...")
#     print(cmd)
#     subprocess.run(cmd, shell=True, check=True)
#     break

Run swarm

In [ ]:
run_perm_swarm(
    swarm_log_dir=SWARM_DIR / "logs",
    cmd_dir=SWARM_DIR / "commands",
    scratch_gb=12,
    n_batch=4,
    process_gb=10,
    num_cpu=16,
    partition="quick",
    swarm_time="0:20:00",
)